# Continuous Glucose Monitoring (CGM) Report

Ambulatory Glucose Profile for the **launched** patient, sourced from the JupyterHealth
Exchange. This notebook is rendered by Voilà as the provider-facing app: the EHR launch
supplies the patient, the cells below compute and draw the report. Code input is hidden
in the rendered view (`strip_sources`), so only the report shows.

In [ ]:
# ============================================================================
# SCAFFOLD — launch context → JHE token → MRN → this patient's glucose frame.
# Usually untouched. Edit the ANALYTICS cells below, not this one.
# ============================================================================
import pandas as pd
from jupyterhealth_client import Code
from provider_app import launch_context, jhe_auth, patient_resolver

pd.options.mode.chained_assignment = None

ctx = launch_context.current()                       # SMART launch (Medplum) → patient MRN
client = jhe_auth.client_for_launch(ctx)             # JHE client (static $JHE_TOKEN for now)
patient_id = patient_resolver.resolve_patient(ctx.patient_mrn, client)   # MRN → JHE patient id

df = client.list_observations_df(
    patient_id=patient_id, code=Code.BLOOD_GLUCOSE, limit=10_000
)

# --- DEV-ONLY fallback (no live launch in Jupyter Lab) ---------------------
# When iterating on visuals outside a SMART launch, comment the three lines
# above and hardcode a known JHE patient instead (needs $JHE_URL/$JHE_TOKEN):
#   from jupyterhealth_client import JupyterHealthClient
#   client = JupyterHealthClient()
#   patient_id = 40006   # May Nguyen, iglu CGM test patient on jhe.fly.dev
#   df = client.list_observations_df(patient_id=patient_id, code=Code.BLOOD_GLUCOSE, limit=10_000)

# Tidy CGM frame, normalized to mg/dL. JHE datasets label the unit variously
# ("mg/dL", "MGDL", "mg/dl"); convert mmol/L (×18.0182) if present, else treat as mg/dL.
if df.empty:
    cgm = df
else:
    cgm = df.loc[:, ["blood_glucose_value", "effective_time_frame_date_time_local"]].copy()
    _unit = df["blood_glucose_unit"].astype(str).str.upper().str.replace(r"[^A-Z]", "", regex=True)
    cgm.loc[_unit.values == "MMOLL", "blood_glucose_value"] *= 18.0182
    cgm["effective_time_frame_date_time_local"] = pd.to_datetime(
        cgm["effective_time_frame_date_time_local"]
    )
    cgm = cgm.sort_values("effective_time_frame_date_time_local")

In [ ]:
# ======== PATIENT HEADER (demographics from the EHR) ========
import requests as _rq
from IPython.display import Markdown, display


def md(text):
    display(Markdown(text))


# Who the patient is comes from the EHR's FHIR Patient (the system of record for
# demographics); the device data comes from JHE. ctx carries the EHR token + FHIR base.
try:
    _p = _rq.get(
        f"{ctx.fhir_base.rstrip('/')}/Patient/{ctx.fhir_patient_id}",
        headers={"Authorization": f"Bearer {ctx.access_token}"},
        timeout=15,
    ).json()
    _nm = (_p.get("name") or [{}])[0]
    _name = (" ".join(_nm.get("given", [])) + " " + _nm.get("family", "")).strip() or "(unnamed)"
    _dob = _p.get("birthDate", "—")
    _sex = (_p.get("gender") or "—").title()
    _contact = next((t.get("value") for t in _p.get("telecom", []) if t.get("value")), "")
except Exception:
    _name, _dob, _sex, _contact = "(demographics unavailable)", "—", "—", ""

try:
    me = client.get_user()
    practitioner = f"{me['firstName']} {me['lastName']} ({me['email']})"
except Exception:
    practitioner = "(unknown)"

md(f"## {_name}")
_line = f"**DOB** {_dob}  ·  **Sex** {_sex}  ·  **MRN** `{ctx.patient_mrn}`  ·  **JHE patient** {patient_id}"
if _contact:
    _line += f"  ·  {_contact}"
md(_line)
md(f"Reviewing practitioner: **{practitioner}**")
if cgm.empty:
    md("> **No glucose data for this patient.** Confirm the patient is enrolled and consented.")
else:
    span = (
        f"{cgm.effective_time_frame_date_time_local.min():%Y-%m-%d} → "
        f"{cgm.effective_time_frame_date_time_local.max():%Y-%m-%d}"
    )
    md(f"{len(cgm):,} glucose readings · {span}")

In [ ]:
# ======== GLYCEMIC METRICS (GMI, CV, Time-in-Range, GRI) ========
if not cgm.empty:
    g = cgm.blood_glucose_value
    total = len(g)
    mean_glucose = g.mean()
    cv = g.std() / mean_glucose * 100
    gmi = 3.31 + 0.02392 * mean_glucose

    vlow = (g < 54).mean() * 100
    low = ((g >= 54) & (g < 70)).mean() * 100
    target = ((g >= 70) & (g <= 180)).mean() * 100
    high = ((g > 180) & (g <= 250)).mean() * 100
    vhigh = (g > 250).mean() * 100

    # Glycemia Risk Index (0 best → 100 worst), capped at 100.
    gri = min(3.0 * (vlow + 0.8 * low) + 1.6 * (vhigh + 0.5 * high), 100)

    md(
        "| Metric | Value |\n|---|---|\n"
        f"| Mean glucose | {mean_glucose:.0f} mg/dL |\n"
        f"| Glucose Management Indicator (GMI) | {gmi:.1f}% |\n"
        f"| Coefficient of variation (CV) | {cv:.0f}% |\n"
        f"| Time in range (70–180) | {target:.0f}% |\n"
        f"| Glycemia Risk Index (GRI) | {gri:.0f} |"
    )

In [ ]:
# ======== AMBULATORY GLUCOSE PROFILE (AGP) ========
# All readings collapsed onto a single 24-hour day, shown as percentile bands.
if not cgm.empty:
    import matplotlib.pyplot as plt

    t = cgm.effective_time_frame_date_time_local
    cgm["minute_of_day"] = t.dt.hour * 60 + t.dt.minute
    cgm["time_bin"] = (cgm.minute_of_day // 5) * 5

    agp = (
        cgm.groupby("time_bin").blood_glucose_value
        .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
        .unstack()
    )
    agp.columns = ["p5", "p25", "p50", "p75", "p95"]
    agp = agp.reset_index()
    agp["hour"] = agp.time_bin / 60

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.axhspan(70, 180, color="#CDECCD", alpha=0.5, label="Target 70–180")
    ax.fill_between(agp.hour, agp.p5, agp.p95, color="#D8E3E7", alpha=0.6, label="5–95%")
    ax.fill_between(agp.hour, agp.p25, agp.p75, color="#B0C4DE", alpha=0.7, label="25–75%")
    ax.plot(agp.hour, agp.p50, color="green", lw=2, label="Median")
    ax.set_title("Ambulatory Glucose Profile (AGP)")
    ax.set_xlabel("Time of day")
    ax.set_ylabel("Glucose (mg/dL)")
    ax.set_xticks([0, 6, 12, 18, 24])
    ax.set_xticklabels(["12 AM", "6 AM", "12 PM", "6 PM", "12 AM"])
    ax.set_xlim(0, 24)
    ax.set_ylim(0, 300)
    ax.set_yticks([54, 70, 180, 250, 300])
    ax.legend(loc="upper right", ncol=2)
    ax.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# ======== TIME IN RANGE (stacked bar) ========
if not cgm.empty:
    bands = {
        "Very Low (<54)": (vlow, "#a00000"),
        "Low (54–69)": (low, "#f44336"),
        "Target (70–180)": (target, "#9ccc9c"),
        "High (181–250)": (high, "#fdbe85"),
        "Very High (>250)": (vhigh, "#fd8d3c"),
    }
    fig, ax = plt.subplots(figsize=(3, 6))
    bottom = 0
    for label, (pct, color) in bands.items():
        ax.bar(0, pct, bottom=bottom, color=color, width=0.6, edgecolor="white")
        if pct > 3:
            ax.text(0.45, bottom + pct / 2, f"{pct:.0f}%  {label}", va="center", fontsize=9)
        bottom += pct
    ax.set_ylim(0, 100)
    ax.set_xlim(-0.4, 2.2)
    ax.set_xticks([])
    ax.set_ylabel("% of readings")
    ax.set_title("Time in Range")
    plt.show()

<hr style="border:none;border-top:3px solid #d96a1f;margin:48px 0 8px">

# Exploration — what's possible with this data

*Everything above is the standard clinical CGM report. Below is the **same patient**, enriched with the sleep, activity, and overnight-vitals signals also available in the JupyterHealth Exchange — an interactive Plotly + ipywidgets dashboard built entirely in this notebook. It shows only signals the patient actually has.*

In [ ]:
# === CGM SHOWCASE LIBRARY ===
# Inline, self-contained: parse OMH bodies -> aggregate -> Plotly figures + live fetch.
# Voila hides cell sources (strip_sources), so this defines the dashboard without showing code.
import base64
import json

import pandas as pd
import plotly.graph_objects as go

# OMH/IEEE codes verified live against JHE (patient 40006, study 30006).
_SIGNAL_CODES = {
    "sleep": "omh:sleep-episode:1.1",
    "activity": "ieee:physical-activity:1.0",
    "heart_rate": "omh:heart-rate:2.0",
    "respiratory_rate": "omh:respiratory-rate:2.0",
    "oxygen_saturation": "omh:oxygen-saturation:2.0",
}
_GLUCOSE_TIME = "effective_time_frame_date_time_local"
_GLUCOSE_VALUE = "blood_glucose_value"

_GLUCOSE = "#e8820c"
_SLEEP_FILL = "rgba(58,110,165,0.20)"
_ACT_LINE = "rgba(31,158,122,0.55)"
_ACT_FILL = "rgba(31,158,122,0.14)"
_BAND = "rgba(31,158,122,0.12)"


# ---- OMH body parsing ----
def _interval(body):
    return (body.get("effective_time_frame") or {}).get("time_interval") or {}


def _val(body, key):
    return (body.get(key) or {}).get("value")


def parse_sleep_episode(body):
    ti = _interval(body)
    start, end = ti.get("start_date_time"), ti.get("end_date_time")
    if not start or not end:
        return None
    start_ts = pd.Timestamp(start)
    return {"start": start_ts, "end": pd.Timestamp(end), "date": start_ts.strftime("%Y-%m-%d"),
            "tst_sec": _val(body, "total_sleep_time"),
            "eff": _val(body, "sleep_maintenance_efficiency_percentage")}


def parse_activity(body):
    ti = _interval(body)
    start = ti.get("start_date_time")
    if not start:
        return None
    return {"date": pd.Timestamp(start).strftime("%Y-%m-%d"),
            "steps": _val(body, "base_movement_quantity"),
            "kcal": _val(body, "kcal_burned"), "distance_m": _val(body, "distance")}


def parse_overnight_vital(body, value_key):
    ti = _interval(body)
    start = ti.get("start_date_time")
    value = _val(body, value_key)
    if not start or value is None:
        return None
    return {"date": pd.Timestamp(start).strftime("%Y-%m-%d"), "value": value}


# ---- Live fetch (single unfiltered call, bucket by code, decode base64 OMH body) ----
def _code_of(record):
    coding = (record.get("code") or {}).get("coding") or [{}]
    return coding[0].get("code")


def _decode_body(record):
    data = (record.get("valueAttachment") or {}).get("data")
    if not data:
        return None
    try:
        return json.loads(base64.b64decode(data)).get("body")
    except Exception:
        return None


def _observations_by_code(client, patient_id):
    buckets = {}
    for rec in client.list_observations(patient_id=patient_id, limit=50_000):
        body = _decode_body(rec)
        if body is None:
            continue
        buckets.setdefault(_code_of(rec), []).append(body)
    return buckets


def fetch_signals(patient_id, client):
    try:
        by_code = _observations_by_code(client, patient_id)
    except Exception:
        by_code = {}

    def bodies(name):
        return by_code.get(_SIGNAL_CODES[name], [])

    sleep = [r for b in bodies("sleep") if (r := parse_sleep_episode(b))]
    activity = [r for b in bodies("activity") if (r := parse_activity(b))]
    vitals = {}
    for short, name in (("hr", "heart_rate"), ("rr", "respiratory_rate"), ("spo2", "oxygen_saturation")):
        vitals[short] = [r for b in bodies(name) if (r := parse_overnight_vital(b, name))]
    return {"sleep": sleep, "activity": activity, "vitals": vitals}


# ---- Aggregation ----
def time_in_range(values, lo, hi):
    s = pd.Series(list(values), dtype="float64")
    if s.empty:
        return 0.0
    return float(((s >= lo) & (s <= hi)).mean() * 100)


def _glucose_frame(cgm):
    # The source column is wall-clock local time; we treat it as UTC for internal
    # consistency (the window boundaries below are built the same way). Renders match.
    if cgm is None or cgm.empty:
        return pd.DataFrame(columns=["time", "value"])
    return pd.DataFrame({
        "time": pd.to_datetime(cgm[_GLUCOSE_TIME], utc=True),
        "value": cgm[_GLUCOSE_VALUE].astype("float64"),
    }).sort_values("time").reset_index(drop=True)


def build_payload(cgm, sleep, activity, vitals):
    glucose = _glucose_frame(cgm)
    # Parser dicts carry exactly these keys; columns= fixes order (empty-safe too).
    sleep_df = pd.DataFrame(sleep, columns=["date", "start", "end", "tst_sec", "eff"])
    act_df = pd.DataFrame(activity, columns=["date", "steps", "kcal", "distance_m"])

    vit = {}
    for short, rows in (("hr", vitals.get("hr", [])), ("rr", vitals.get("rr", [])), ("spo2", vitals.get("spo2", []))):
        vit[short] = pd.Series({r["date"]: r["value"] for r in rows}, dtype="float64")
    vitals_df = pd.DataFrame(vit)
    if not vitals_df.empty:
        vitals_df.index.name = "date"

    if not act_df.empty:
        days = pd.to_datetime(act_df["date"], utc=True)
        start = days.min().normalize()
        end = days.max().normalize() + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
    elif not glucose.empty:
        start, end = glucose["time"].min(), glucose["time"].max()
    else:
        start = end = pd.Timestamp("1970-01-01T00:00:00Z")

    if not glucose.empty:
        glucose = glucose[(glucose["time"] >= start) & (glucose["time"] <= end)].reset_index(drop=True)

    return {"glucose": glucose, "sleep": sleep_df, "activity": act_df, "vitals": vitals_df,
            "window": {"start": start, "end": end},
            "has": {"glucose": not glucose.empty, "sleep": not sleep_df.empty,
                    "activity": not act_df.empty, "vitals": not vitals_df.empty}}


def summary_stats(payload):
    g = payload["glucose"]["value"] if not payload["glucose"].empty else pd.Series(dtype="float64")
    act, sleep = payload["activity"], payload["sleep"]
    mean_glucose = float(g.mean()) if not g.empty else None
    cv = float(g.std() / g.mean() * 100) if (not g.empty and g.mean() != 0 and not pd.isna(g.std())) else None
    return {
        "mean_glucose": round(mean_glucose) if mean_glucose is not None else None,
        "tir": round(time_in_range(g, 70, 180)) if not g.empty else None,
        "cv": round(cv) if cv is not None else None,
        "avg_steps": round(float(act["steps"].mean())) if not act.empty and act["steps"].notna().any() else None,
        "avg_kcal": round(float(act["kcal"].mean())) if not act.empty and act["kcal"].notna().any() else None,
        "avg_sleep_eff": round(float(sleep["eff"].mean()), 1) if not sleep.empty and sleep["eff"].notna().any() else None,
    }


# ---- Figures ----
def available_days(payload):
    g = payload["glucose"]
    if g.empty:
        return []
    return sorted(g["time"].dt.strftime("%Y-%m-%d").unique().tolist())


def overlay_figure(payload):
    g = payload["glucose"]
    fig = go.Figure()
    if not g.empty:
        fig.add_trace(go.Scatter(x=g["time"], y=g["value"], name="Glucose",
                                 mode="lines", line=dict(color=_GLUCOSE, width=1.6)))
    if payload["has"]["activity"]:
        xs, ys = [], []
        for _, r in payload["activity"].dropna(subset=["kcal"]).iterrows():
            d0 = pd.Timestamp(r["date"] + "T00:00:00Z")
            xs += [d0, d0 + pd.Timedelta(hours=24)]
            ys += [r["kcal"], r["kcal"]]
        if xs:
            fig.add_trace(go.Scatter(x=xs, y=ys, name="Calories", yaxis="y2", mode="lines",
                                     line=dict(color=_ACT_LINE, width=1, shape="hv"),
                                     fill="tozeroy", fillcolor=_ACT_FILL))
    if payload["has"]["sleep"]:
        for _, s in payload["sleep"].iterrows():
            if pd.isna(s["start"]) or pd.isna(s["end"]):
                continue
            # Plotly converts tz-aware datetimes for TRACE data but not for layout shape
            # coords, so a tz-aware x0/x1 lands off the (tz-naive) date axis and the band
            # never renders. Drop the tz to match the axis.
            x0 = s["start"].tz_localize(None) if s["start"].tzinfo else s["start"]
            x1 = s["end"].tz_localize(None) if s["end"].tzinfo else s["end"]
            fig.add_vrect(x0=x0, x1=x1, fillcolor=_SLEEP_FILL, line_width=0, layer="below")
        # vrect shapes don't show in the legend; add a labeled proxy so the bands read as sleep.
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers", name="Sleep window",
            marker=dict(size=11, color="rgba(58,110,165,0.55)", symbol="square"),
            hoverinfo="skip"))
    fig.update_layout(
        margin=dict(l=50, r=50, t=30, b=40), height=460, hovermode="x unified",
        xaxis=dict(type="date"),
        yaxis=dict(title="glucose mg/dL", range=[50, 300]),
        yaxis2=dict(title="kcal/day", overlaying="y", side="right", range=[0, 1800], showgrid=False),
        legend=dict(orientation="h"), template="plotly_white")
    return fig


def day_figure(payload, day, lo, hi):
    g = payload["glucose"]
    d = g[g["time"].dt.strftime("%Y-%m-%d") == day] if not g.empty else g
    fig = go.Figure()
    tir = time_in_range(d["value"], lo, hi) if not d.empty else 0.0
    if not d.empty:
        fig.add_trace(go.Scatter(x=d["time"], y=d["value"], name="Glucose", mode="lines",
                                 line=dict(color=_GLUCOSE, width=2), fill="tozeroy",
                                 fillcolor="rgba(232,130,12,0.10)"))
        fig.add_hrect(y0=lo, y1=hi, fillcolor=_BAND, line_width=0, layer="below")
        if len(d) >= 2:  # low/peak only meaningful with >=2 readings
            lo_i, hi_i = d["value"].idxmin(), d["value"].idxmax()
            fig.add_trace(go.Scatter(
                x=[d.loc[lo_i, "time"], d.loc[hi_i, "time"]],
                y=[d.loc[lo_i, "value"], d.loc[hi_i, "value"]],
                mode="markers+text", text=["low", "peak"], textposition="bottom center",
                marker=dict(size=8, color=["#d64545", "#c77d1a"]), showlegend=False))
    fig.update_layout(
        title=f"{day} — {tir:.0f}% in range ({lo}–{hi})",
        margin=dict(l=50, r=20, t=40, b=40), height=420, hovermode="x unified",
        yaxis=dict(title="mg/dL", range=[50, max(190, (d['value'].max() + 15) if not d.empty else 190)]),
        template="plotly_white")
    return fig


def night_figures(payload):
    g = payload["glucose"]
    if g.empty:
        return []
    t = g["time"]
    hour = t.dt.hour + t.dt.minute / 60.0
    anchor = t.dt.strftime("%Y-%m-%d")
    morning = hour < 12
    anchor = anchor.where(~morning, (t - pd.Timedelta(days=1)).dt.strftime("%Y-%m-%d"))
    rel = hour.where(hour >= 18, hour + 24) - 18  # 18:00->0, 00:00->6, 06:00->12, noon->18
    work = pd.DataFrame({"anchor": anchor, "rel": rel, "value": g["value"], "hour": hour})
    work = work[(work["hour"] >= 18) | (work["hour"] < 12)]
    out = []
    for key in sorted(work["anchor"].unique()):
        seg = work[work["anchor"] == key].sort_values("rel")
        if len(seg) < 24:  # need ~2h+ of overnight coverage to be worth plotting
            continue
        fig = go.Figure(go.Scatter(x=seg["rel"], y=seg["value"], mode="lines",
                                   line=dict(color=_GLUCOSE, width=1.8), fill="tozeroy",
                                   fillcolor="rgba(232,130,12,0.10)"))
        fig.update_layout(height=160, margin=dict(l=30, r=8, t=8, b=20), showlegend=False,
                          template="plotly_white",
                          xaxis=dict(tickvals=[0, 6, 12, 18], ticktext=["18:00", "00:00", "06:00", "12:00"]),
                          yaxis=dict(range=[50, max(190, seg["value"].max() + 12)]))
        out.append((key, fig))
    return out


In [ ]:
# ======== SHOWCASE — fetch multi-signal data & build payload ========
showcase_payload = None
showcase_note = None
if access_note:
    showcase_note = "Showcase hidden — no authorized JupyterHealth access for this patient."
elif cgm.empty:
    showcase_note = "No glucose data, so there's nothing to enrich for this patient."
else:
    try:
        _sig = fetch_signals(patient_id, client)
        showcase_payload = build_payload(cgm, _sig["sleep"], _sig["activity"], _sig["vitals"])
    except Exception as _e:
        showcase_note = f"Couldn't build the showcase — {type(_e).__name__}: {_e}"

if showcase_note:
    md(f"> {showcase_note}")

In [ ]:
# ======== SHOWCASE — summary stats strip ========
if showcase_payload is not None:
    from IPython.display import HTML
    _s = summary_stats(showcase_payload)
    _cells = [
        ("Mean glucose", _s["mean_glucose"], "mg/dL"),
        ("Time in range", _s["tir"], "% 70–180"),
        ("Glucose CV", _s["cv"], "%"),
        ("Avg steps", f"{_s['avg_steps']:,}" if _s["avg_steps"] else None, "per day"),
        ("Avg calories", f"{_s['avg_kcal']:,}" if _s["avg_kcal"] else None, "kcal/day"),
        ("Avg sleep eff.", _s["avg_sleep_eff"], "%"),
    ]
    _html = "".join(
        "<div style='flex:1;min-width:120px;padding:10px 14px;border:1px solid #e2ddd3;border-radius:8px'>"
        f"<div style='font-size:22px;font-weight:600'>{v} <small style='color:#6b7a85;font-size:12px'>{u}</small></div>"
        f"<div style='font-family:monospace;font-size:10px;letter-spacing:.1em;text-transform:uppercase;color:#6b7a85'>{lab}</div></div>"
        for (lab, v, u) in _cells if v is not None
    )
    display(HTML(f"<div style='display:flex;gap:8px;flex-wrap:wrap;margin:12px 0'>{_html}</div>"))

In [ ]:
# ======== SHOWCASE — interactive views (Master Overlay / Anatomy of a Day / Overnight) ========
if showcase_payload is not None:
    import ipywidgets as W
    import plotly.graph_objects as go
    from IPython.display import display
    _p = showcase_payload

    # Tab 1 — Master Overlay
    overlay_w = go.FigureWidget(overlay_figure(_p))

    # Tab 2 — Anatomy of a Day (day picker + live normal-range slider)
    _days = available_days(_p)
    if _days:
        _dd = W.Dropdown(options=_days, value=_days[0], description="Day")
        _rng = W.IntRangeSlider(value=[70, 180], min=40, max=300, step=5,
                                description="Range", continuous_update=False)
        _day_out = W.Output()

        def _render_day(*_):
            _day_out.clear_output(wait=True)
            with _day_out:
                display(go.FigureWidget(day_figure(_p, _dd.value, *_rng.value)))

        _dd.observe(_render_day, "value")
        _rng.observe(_render_day, "value")
        _render_day()
        anatomy = W.VBox([W.HBox([_dd, _rng]), _day_out])
    else:
        anatomy = W.HTML("<em>No daily glucose to drill into.</em>")

    # Tab 3 — Overnight small multiples with overnight vitals
    _nights = night_figures(_p)
    if _nights:
        _v = _p["vitals"]
        _cards = []
        for _label, _fig in _nights:
            _cap = ""
            if not _v.empty and _label in _v.index:
                _row = _v.loc[_label]
                _bits = [f"{_row[k]:.0f} {u}" for k, u in (("hr", "bpm"), ("rr", "rr"), ("spo2", "%SpO₂"))
                         if k in _v.columns and pd.notna(_row.get(k))]
                _cap = "  ·  ".join(_bits)
            _cards.append(W.VBox([W.HTML(f"<b>{_label} night</b><br><small>{_cap}</small>"),
                                  go.FigureWidget(_fig)]))
        overnight = W.GridBox(_cards, layout=W.Layout(
            grid_template_columns="repeat(auto-fill, minmax(260px, 1fr))", grid_gap="14px"))
    else:
        overnight = W.HTML("<em>No overnight glucose windows.</em>")

    _tabs = W.Tab(children=[overlay_w, anatomy, overnight])
    for _i, _t in enumerate(["Master Overlay", "Anatomy of a Day", "Overnight"]):
        _tabs.set_title(_i, _t)
    display(_tabs)